<a href="https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (confirmed, not switched): Freestyle — "Confound or Cause?"**

ML-03 and ML-04 asked whether an apparent effect survives its confounds. This week that same
question is pointed at a *rule*: FlyRank's flags lean on signals that look obvious. Before I
encode one into a queue that tells a human what to open first, I check whether the two signals
my rule leans on are real — and one of them is not.

Everything below runs on `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients),
the same slice and the same label the Week-5 model will use:
`is_declining_label = (trend_direction == "down")`.

The rule is fully deterministic — no fitted weights, no random state.

In [1]:
import os, sys, json
import pandas as pd, numpy as np

# Land on the repo root from anywhere: a fresh Colab session, notebooks/, or a repeat Run All.
# Idempotent on purpose -- the test is "can I see data/raw", never "does a folder name exist",
# so re-running this cell inside the repo cannot clone a nested copy of it.
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-capstone"

if IN_COLAB and not os.path.isdir("data/raw"):
    if not os.path.isdir(REPO_DIR):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/ahmadhalawanii/{REPO_DIR}", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

assert os.path.isdir("data/raw"), f"repo root not found from {os.getcwd()}"
print("working dir:", os.getcwd())
os.makedirs("work/outputs", exist_ok=True)
pd.set_option("display.width", 170)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values
BASE_RATE = y.mean()

print("rows:", len(df), "| clients:", df["client_id"].nunique())
print(f"base rate (share declining): {BASE_RATE:.3f}")

working dir: /content/flyrank-ml-capstone
rows: 30000 | clients: 32
base rate (share declining): 0.542


## 1. My rule and its reason codes

### The rule idea, in plain words

> A page is worth a title-and-meta review if it **already ranks in the top 20** (the impressions
> are there to convert), it has **enough impressions for a click-through rate to mean anything**,
> and its **CTR sits below the median CTR of pages at its own position tier**. Rank those by how
> far below that median they are.

That rule leans on two signals, and both sit behind flags FlyRank actually ships:

- **Signal 1 — staleness** (behind the refresh flags): stale pages decline more.
- **Signal 2 — CTR relative to position** (behind the CTR-fix logic): a page ranking well but
  under-clicking for its position is in trouble.

I check both *before* encoding anything. Two gotchas from `docs/data-dictionary.md` apply
throughout: `ctr` is a ×100 percentage (`0.23` means 0.23%), and `avg_position = 0` means
"no position data", not rank zero — those rows are excluded, never treated as rank 1.

### Signal 1 — staleness → decline. Bucket table, with n.

One extra column matters here. The label is `trend_direction == "down"`, which the dictionary
defines as last-30d impressions more than 20% below prev-30d. When `impressions_prev_30d = 0`
there is no denominator, so the row is scored `new` or `flat` — it **can never be labelled
`down`**, whatever it does. So I print the share of each bucket that is even *eligible* to carry
the positive label, and then repeat the table on the eligible rows only.

In [2]:
df["label_eligible"] = df["impressions_prev_30d"] > 0   # diagnostic only -- NOT a rule input

sig1_raw = (df.groupby("freshness_tier")
              .agg(n=("is_declining_label", "size"),
                   declining_rate=("is_declining_label", "mean"),
                   pct_label_eligible=("label_eligible", "mean"))
              .round(3))
print("SIGNAL 1 -- decline rate by freshness_tier (all rows)")
print(sig1_raw.to_string(), "\n")

sig1_elig = (df[df.label_eligible].groupby("freshness_tier")
               .agg(n=("is_declining_label", "size"),
                    declining_rate=("is_declining_label", "mean"))
               .round(3))
print("SIGNAL 1 -- same table, rows where the label CAN be 'down' (impressions_prev_30d > 0)")
print(sig1_elig.to_string(), "\n")

raw_spread  = sig1_raw["declining_rate"].max()  - sig1_raw["declining_rate"].min()
elig_spread = sig1_elig["declining_rate"].max() - sig1_elig["declining_rate"].min()
print(f"spread across buckets -- raw: {raw_spread:.3f}   label-eligible only: {elig_spread:.3f}")

SIGNAL 1 -- decline rate by freshness_tier (all rows)
                    n  declining_rate  pct_label_eligible
freshness_tier                                           
0-30            20480           0.511               0.852
181+              174           0.471               0.764
31-90             175           0.589               0.954
91-180           9171           0.611               0.966 

SIGNAL 1 -- same table, rows where the label CAN be 'down' (impressions_prev_30d > 0)
                    n  declining_rate
freshness_tier                       
0-30            17454           0.600
181+              133           0.617
31-90             167           0.617
91-180           8858           0.633 

spread across buckets -- raw: 0.140   label-eligible only: 0.033


**Verdict: FALSE.**

The raw table looks like a staleness gradient: 0.511 declining in the freshest bucket vs 0.611 in
`91-180` — a 0.140 spread across the four buckets, enough to justify a refresh flag. It doesn't
survive. 14.8% of the
`0-30` bucket cannot carry the `down` label at all (no prior-30-day impressions), against 3.4% of
`91-180` — those structurally-zero rows sit in the denominator of the fresh bucket and drag its
rate down. On rows where the label can actually be `down`, the spread collapses from 0.140 to 0.033 and
stops being monotone: `181+` (0.617) is no worse than `31-90` (0.617), and both sit under
`91-180` (0.633).

So the staleness gradient is mostly a **labelability artifact**, not a decline signal. Measured on
this slice, staleness does not separate declining pages at anything like the strength a refresh
flag assumes. **This is what saved the rule** — my first draft ranked by "stale and visible", and
that queue would have been ordered by an artifact of who had a prior-window baseline.

### Signal 2 — CTR below the median for its own position tier. Bucket table, with n.

The comparison has to be *within* position tier: median CTR runs 0.23% on page 1 but 0.15% in the
striking-distance band, so one global CTR threshold (the starter's `ctr < 0.5`) compares pages to
a bar that means different things at different ranks. The dictionary also warns that tier metrics
need a volume floor, so this is measured on pages with `avg_position` in (0, 20] and at least 100
impressions in 90 days.

In [3]:
FLOOR_IMPRESSIONS = 100
POSITION_MAX     = 20

visible = df[(df.avg_position > 0) & (df.avg_position <= POSITION_MAX)
             & (df.impressions_90d >= FLOOR_IMPRESSIONS)].copy()

TIER_MEDIAN_CTR = visible.groupby("position_tier")["ctr"].median()
print("median ctr by position_tier (visible slice, n = %d):" % len(visible))
print(TIER_MEDIAN_CTR.round(3).to_string(), "\n")

visible["ctr_below_tier_median"] = (visible.ctr < visible.position_tier.map(TIER_MEDIAN_CTR)).astype(int)
sig2 = (visible.groupby(["position_tier", "ctr_below_tier_median"])
        .agg(n=("is_declining_label", "size"), declining_rate=("is_declining_label", "mean"))
        .round(3))
print("SIGNAL 2 -- decline rate by (position tier x below-tier-median CTR)")
print(sig2.to_string())

median ctr by position_tier (visible slice, n = 15091):
position_tier
page_1      0.230
page_3_5    0.155
striking    0.150
top_3       0.190 

SIGNAL 2 -- decline rate by (position tier x below-tier-median CTR)
                                        n  declining_rate
position_tier ctr_below_tier_median                      
page_1        0                      4371           0.535
              1                      4262           0.680
page_3_5      0                        11           0.364
              1                        11           0.636
striking      0                      3036           0.573
              1                      2867           0.683
top_3         0                       269           0.643
              1                       264           0.871


**Verdict: CONFIRMED (on the three tiers with usable n).**

| tier | at/above tier median | below tier median | gap | n |
|---|---|---|---|---|
| `top_3` | 0.643 | 0.871 | **+22.8 pts** | 533 |
| `page_1` | 0.535 | 0.680 | **+14.5 pts** | 8,633 |
| `striking` | 0.573 | 0.683 | **+11.0 pts** | 5,903 |
| `page_3_5` | 0.364 | 0.636 | +27.2 pts | **22 — ignore** |

Same direction, same rough size, on every tier where there are enough rows to read. `page_3_5` is
listed for completeness only: the `avg_position <= 20` gate leaves 22 rows in it, 11 per side, so
its gap is noise and I don't lean on it.

**The rule, as encoded:** score is the CTR shortfall itself, in percentage points —
`tier_median_ctr - ctr`. Not the shortfall multiplied by impressions. That distinction turns out to
matter a great deal, and section 4 shows what happens when you get it backwards.

- **score:** `ctr_gap_pp = tier_median_ctr − ctr`, floored at 0, and 0 for anything not eligible
- **reason code:** `ctr_below_position_tier_median` (else `not_flagged`)
- **action label:** `review_title_and_meta` (else `no_action`)
- **tie-break:** `impressions_90d` descending — declared here, before any precision number is
  computed, on the ordinary grounds that between two pages under-clicking by the same amount, the
  one seen by more people is the one a human should open first

## 2. Build the ranked queue (writes the CSV)

One rule, one reason code, one action label, ranked over all 30,000 rows. `precision@K` is printed
against the base rate — 0.542 here, so a queue has to clear that to have earned anything.

In [4]:
RULE_INPUTS = ["avg_position", "position_tier", "ctr", "impressions_90d"]

tier_med  = df["position_tier"].map(TIER_MEDIAN_CTR)
eligible  = ((df.avg_position > 0) & (df.avg_position <= POSITION_MAX)
             & (df.impressions_90d >= FLOOR_IMPRESSIONS) & tier_med.notna())
ctr_gap   = (tier_med - df["ctr"]).clip(lower=0).round(3)

df["tier_median_ctr"] = tier_med
df["ctr_gap_pp"]      = np.where(eligible, ctr_gap, 0.0)
df["baseline_score"]  = np.where(eligible & (ctr_gap > 0), ctr_gap, 0.0)
flagged               = df.baseline_score > 0
df["reason_code"]     = np.where(flagged, "ctr_below_position_tier_median", "not_flagged")
df["action"]          = np.where(flagged, "review_title_and_meta", "no_action")

queue = df.sort_values(["baseline_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["queue_rank"] = np.arange(1, len(queue) + 1)

def precision_at_k(labels, k):
    return np.asarray(labels)[:k].mean()

ks = [10, 20, 50, 100, 500]
prec = {k: float(precision_at_k(queue.is_declining_label.values, k)) for k in ks}

print(f"eligible rows: {int(eligible.sum())}   flagged: {int(flagged.sum())} ({flagged.mean():.1%} of all pages)")
print(f"decline rate among flagged pages: {df.loc[flagged,'is_declining_label'].mean():.3f}   (base rate {BASE_RATE:.3f})\n")
for k in ks:
    print(f"precision@{k:<4d} {prec[k]:.3f}    base rate {BASE_RATE:.3f}    lift {prec[k]-BASE_RATE:+.3f}")

eligible rows: 15091   flagged: 7404 (24.7% of all pages)
decline rate among flagged pages: 0.688   (base rate 0.542)

precision@10   0.800    base rate 0.542    lift +0.258
precision@20   0.800    base rate 0.542    lift +0.258
precision@50   0.860    base rate 0.542    lift +0.318
precision@100  0.880    base rate 0.542    lift +0.338
precision@500  0.824    base rate 0.542    lift +0.282


In [5]:
OUT_COLS = ["queue_rank", "content_id", "baseline_score", "reason_code", "action",
            "position_tier", "avg_position", "ctr", "tier_median_ctr", "ctr_gap_pp",
            "impressions_90d", "clicks_90d", "sessions_90d", "content_type",
            "word_count", "content_age_days", "days_since_last_update"]

CSV_PATH = "work/outputs/baseline_action_score.csv"
queue[OUT_COLS].to_csv(CSV_PATH, index=False)
print("wrote", CSV_PATH, "|", len(queue), "rows x", len(OUT_COLS), "cols")
print("(gitignored by design -- the CI leak-guard blocks data files; this cell regenerates it)")

metrics = {
    "notebook": "w04_baseline_score.ipynb",
    "assignment": "ML-07",
    "lane": "Freestyle: Confound or Cause?",
    "data_slice": "data/raw/content_refresh_anonymized.csv (30,000 rows, 32 clients)",
    "label": "is_declining_label = (trend_direction == 'down')",
    "base_rate": round(float(BASE_RATE), 4),
    "rule": {
        "plain_words": "top-20 rank, >=100 impressions in 90d, CTR below the median CTR of its own position tier",
        "score": "tier_median_ctr - ctr, in percentage points",
        "reason_code": "ctr_below_position_tier_median",
        "action": "review_title_and_meta",
        "tie_break": "impressions_90d descending",
        "inputs": RULE_INPUTS,
        "position_max": POSITION_MAX,
        "impression_floor": FLOOR_IMPRESSIONS,
        "tier_median_ctr": {k: float(v) for k, v in TIER_MEDIAN_CTR.round(3).items()},
        "deterministic": True,
    },
    "signal_verdicts": {
        "staleness_behind_refresh_flags": "FALSE",
        "ctr_below_position_tier_median_behind_ctr_fix": "CONFIRMED",
    },
    "n_eligible": int(eligible.sum()),
    "n_flagged": int(flagged.sum()),
    "precision_at_k": {str(k): round(v, 4) for k, v in prec.items()},
}
with open("work/outputs/baseline_action_score_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/baseline_action_score_metrics.json  (committed -- the receipt for every number above)")

wrote work/outputs/baseline_action_score.csv | 30000 rows x 17 cols
(gitignored by design -- the CI leak-guard blocks data files; this cell regenerates it)
wrote work/outputs/baseline_action_score_metrics.json  (committed -- the receipt for every number above)


## 3. Top-20 review

The card asks for ten. The skeleton header and the `building-baselines` skill both say twenty, and
twenty is where bad logic shows itself — so this reviews twenty.

For each row: the action, the reason code, a confidence note, why it's there, and what would make
it wrong. The confidence note and the "what would make it wrong" line are both generated from
explicit checks against the row rather than typed by hand, so they stay honest when the data
changes — first applicable check wins, one line each.

In [6]:
def what_would_make_it_wrong(r):
    if r.clicks_90d == 0 and r.impressions_90d >= 1000:
        return "zero clicks on 1k+ impressions -- more likely broken click tracking than a bad title"
    if r.impressions_90d >= 1000 and r.sessions_90d < 10:
        return "GSC impressions and GA4 sessions disagree by orders of magnitude -- one system isn't measuring this page"
    if pd.isna(r.word_count):
        return "no word_count recorded -- can't confirm there is a real page here to rewrite"
    if pd.isna(r.search_volume) or r.search_volume == 0:
        return "no keyword demand recorded -- tier median may not describe this page's traffic mix"
    if abs(r.avg_position - 10) <= 1 or abs(r.avg_position - 3) <= 1:
        return "sits on a tier boundary -- a small position move swaps which median it's compared against"
    if r.content_age_days < 120:
        return "page is young -- CTR may still be settling, not underperforming"
    return "CTR is real but low -- the title may already be the best available for this query"

TIE_MAX   = queue.baseline_score.max()
N_TIED    = int((queue.baseline_score == TIE_MAX).sum())

def confidence_note(r):
    if r.baseline_score == TIE_MAX and N_TIED > 1:
        return f"LOW -- tied with {N_TIED-1:,} other pages at the identical score; its place in the queue is the tie-break's doing, not the rule's"
    if r.clicks_90d == 0:
        return "LOW -- a 0.00% CTR is a measurement, not a rate; 'nobody clicked' and 'clicks not recorded' look identical here"
    if r.impressions_90d < 500:
        return "LOW -- small impression base, so the CTR estimate is noisy"
    if r.ctr_gap_pp < 0.05:
        return "MEDIUM -- real but shallow shortfall against the tier median"
    return "HIGH -- clear shortfall on a stable impression base"

top10 = queue.head(20).copy()
top10["why_it_is_here"] = top10.apply(
    lambda r: f"ranks {r.avg_position:.1f} ({r.position_tier}), CTR {r.ctr:.2f}% vs tier median "
              f"{r.tier_median_ctr:.2f}% -> {r.ctr_gap_pp:.2f}pp short on {int(r.impressions_90d):,} impressions", axis=1)
top10["what_would_make_it_wrong"] = top10.apply(what_would_make_it_wrong, axis=1)
top10["confidence"] = top10.apply(confidence_note, axis=1)

for _, r in top10.iterrows():
    print(f"#{int(r.queue_rank):>2}  {r.content_id}  [{r.action}]  reason: {r.reason_code}")
    print(f"     why:        {r.why_it_is_here}")
    print(f"     confidence: {r.confidence}")
    print(f"     wrong:      {r.what_would_make_it_wrong}")
    print(f"     clicks_90d: {int(r.clicks_90d)}   sessions_90d: {int(r.sessions_90d)}   "
          f"actual label: {'declining' if r.is_declining_label else 'NOT declining'}")
print(f"\n{int(top10.is_declining_label.sum())} of the top 20 are actually declining "
      f"(precision@20 = {prec[20]:.3f}, base rate {BASE_RATE:.3f})")
print(f"distinct scores across the reviewed 20: {sorted(float(v) for v in top10.baseline_score.unique())}")
print(f"reviewed rows with zero clicks in 90d: {int((top10.clicks_90d == 0).sum())} of 20")
print(f"confidence notes issued: {top10.confidence.str.split(' --').str[0].value_counts().to_dict()}")

# 1  content_c8e9d6ab9013  [review_title_and_meta]  reason: ctr_below_position_tier_median
     why:        ranks 9.7 (page_1), CTR 0.00% vs tier median 0.23% -> 0.23pp short on 208,678 impressions
     confidence: LOW -- tied with 1,347 other pages at the identical score; its place in the queue is the tie-break's doing, not the rule's
     wrong:      zero clicks on 1k+ impressions -- more likely broken click tracking than a bad title
     clicks_90d: 0   sessions_90d: 6   actual label: declining
# 2  content_f986bd514b6e  [review_title_and_meta]  reason: ctr_below_position_tier_median
     why:        ranks 6.6 (page_1), CTR 0.00% vs tier median 0.23% -> 0.23pp short on 22,456 impressions
     confidence: LOW -- tied with 1,347 other pages at the identical score; its place in the queue is the tie-break's doing, not the rule's
     wrong:      GSC impressions and GA4 sessions disagree by orders of magnitude -- one system isn't measuring this page
     clicks_90d: 1   sessions_90d: 4  

### Reading my own top twenty

16 of 20 are genuinely declining — precision@20 of 0.800 against a 0.542 base rate. And I don't
trust the list.

Every one of the twenty has a recorded CTR of exactly 0.00%, nineteen have **literally zero clicks
in 90 days**, all twenty are `page_1`, and all twenty carry the *same* score of 0.23. The top row
has 208,678 impressions, 0 clicks, and 6 GA4 sessions. A page ranking at position 9.7 with two
hundred thousand impressions and not one click is not a page with a weak title. It's a page whose
clicks aren't being recorded — a property mismatch, a filtered subdomain, a redirect that GSC and
GA4 disagree about.

Every one of the twenty confidence notes came back LOW, and not one of them is low for a reason
about the *content*. They are low because the score is tied 1,348 ways and because a 0.00% CTR
can't distinguish "nobody clicked" from "clicks not recorded." A hand review that returns twenty
low-confidence rows in a row is not a review that found nothing — it's the review telling me the
queue's whole head is one undifferentiated block.

That is the whole lesson of this baseline. **The metric is right and the action is wrong.** These
pages really are declining, so they score well on precision@10, but the action attached to them —
`review_title_and_meta` — would send a human to rewrite a title on a page whose measurement is
broken. Precision against the label says nothing about whether the recommendation is correct, and a
queue is judged on the second thing. Every row in that top twenty needs a tracking check before
anyone touches the copy.

## 4. Weak picks + leakage check

### Weak pick 1 — the zero-click block

The pattern in the top ten is not confined to the top ten.

In [7]:
fl   = df[flagged]
zero = fl.clicks_90d == 0
print(f"flagged pages: {len(fl)}")
print(f"  with zero clicks in 90d: {int(zero.sum())} ({zero.mean():.1%})")
print(f"  decline rate | zero clicks : {fl.loc[zero,  'is_declining_label'].mean():.3f}  (n={int(zero.sum())})")
print(f"  decline rate | some clicks : {fl.loc[~zero, 'is_declining_label'].mean():.3f}  (n={int((~zero).sum())})")
print(f"\nacross all 30,000 pages, impressions >= 1,000 with zero clicks: "
      f"{int(((df.impressions_90d >= 1000) & (df.clicks_90d == 0)).sum())}")

flagged pages: 7404
  with zero clicks in 90d: 3053 (41.2%)
  decline rate | zero clicks : 0.756  (n=3053)
  decline rate | some clicks : 0.640  (n=4351)

across all 30,000 pages, impressions >= 1,000 with zero clicks: 1149


Zero-click pages decline *more* than the clicking ones, so they pull precision up while making the
queue less actionable — the metric rewards exactly the rows a human cannot act on. Excluding them
is the obvious v2, and I'm declaring it as a **Week-5 hypothesis, not this week's baseline**. I only
know it after seeing the labels, and a baseline you retune once you've seen the answer isn't a
baseline. It stays frozen as scored.

### Weak pick 2 — the composition mistake I almost shipped

My first encoding scored `missed_clicks = impressions × ctr_gap`, which reads like the more
business-shaped metric: "clicks left on the table." Same two ingredients as the rule I shipped,
composed differently. It ranks **below random**.

In [8]:
rejected = np.where(eligible & (ctr_gap > 0), df.impressions_90d * ctr_gap / 100, 0.0)
rej_q    = df.assign(s=rejected).sort_values("s", ascending=False)

print("REJECTED variant -- score = impressions_90d * ctr_gap (\"missed clicks\")")
for k in ks:
    p = rej_q.is_declining_label.head(k).mean()
    print(f"  precision@{k:<4d} {p:.3f}   vs shipped rule {prec[k]:.3f}   vs base rate {BASE_RATE:.3f}")

print("\nwhy: multiplying by impressions hands the ranking to the highest-volume pages,")
print("and the 'excellent' impression tier is the *least* declining group in the data:")
print(df.groupby("impression_tier")
        .agg(n=("is_declining_label","size"), declining_rate=("is_declining_label","mean"))
        .round(3).to_string())

REJECTED variant -- score = impressions_90d * ctr_gap ("missed clicks")
  precision@10   0.500   vs shipped rule 0.800   vs base rate 0.542
  precision@20   0.500   vs shipped rule 0.800   vs base rate 0.542
  precision@50   0.540   vs shipped rule 0.860   vs base rate 0.542
  precision@100  0.460   vs shipped rule 0.880   vs base rate 0.542
  precision@500  0.554   vs shipped rule 0.824   vs base rate 0.542

why: multiplying by impressions hands the ranking to the highest-volume pages,
and the 'excellent' impression tier is the *least* declining group in the data:
                     n  declining_rate
impression_tier                       
excellent         1078           0.462
good              7205           0.586
low              11248           0.454
moderate         10469           0.615


Multiplied in, volume swamps the CTR gap and the queue fills with big stable pages — 0.500 at
K=10, *below* the 0.542 base rate. Used only to break ties within an identical gap, the same column
helps. Same two ingredients, opposite result: what a transparent score does depends on how its
parts are composed, not just which parts are in it.

### Weak pick 3 — the queue has 29 rungs, not 7,404

1,348 pages tie at the maximum score of 0.23, and the entire top 100 lives inside that block — so
precision@10 through precision@100 is decided by the tie-break, not by the rule. Notebook 02 flagged
the same failure on the depth-2 tree.

But "lots of pages have 0.00% CTR" is the symptom, not the cause. `ctr` ships rounded to two
decimals, so `tier_median_ctr − ctr` can only ever land on a handful of values: **7,404 flagged
pages share 29 distinct scores.** The ties are structural — they come from the precision the column
was stored at, not from the pages being genuinely alike.

In [9]:
flag_scores = df.loc[flagged, "baseline_score"]
print(f"flagged pages: {len(flag_scores):,}   distinct score values among them: {flag_scores.nunique()}")
print(f"ctr is stored to {int(df.ctr.map(lambda v: len(str(v).split('.')[-1]) if '.' in str(v) else 0).max())} decimals "
      f"({df.ctr.nunique()} distinct values across all 30,000 rows)\n")
print("largest score groups:")
print(flag_scores.value_counts().sort_index(ascending=False).head(5).to_string())

tie_block = int((df.baseline_score == df.baseline_score.max()).sum())
print(f"\ntop tie block: {tie_block:,} pages -- all page_1: "
      f"{bool((df.loc[df.baseline_score == df.baseline_score.max(), 'position_tier'] == 'page_1').all())}, "
      f"all CTR 0.00: {bool((df.loc[df.baseline_score == df.baseline_score.max(), 'ctr'] == 0).all())}")

flagged pages: 7,404   distinct score values among them: 29
ctr is stored to 2 decimals (401 distinct values across all 30,000 rows)

largest score groups:
baseline_score
0.23    1348
0.22      18
0.21      42
0.20      81
0.19     190

top tie block: 1,348 pages -- all page_1: True, all CTR 0.00: True


**What that means for Week 5, stated now rather than after the fact:** a model producing a
continuous probability will split this tie block by construction. Some of its precision@10 gain
will be resolution the baseline never had, not skill the baseline lacked. The comparison that
survives that objection is precision at larger K, where the ordering stops being tie-break noise —
which is why **precision@50 and precision@100 are the numbers I'm freezing**, and why the sensitivity
band below is part of the baseline rather than a footnote to it.

Below: the band across three tie-breaks — the one declared in section 1, its reverse, and a random
shuffle.

In [10]:
tie_block = int((df.baseline_score == df.baseline_score.max()).sum())
print(f"pages tied at the maximum score ({df.baseline_score.max():.2f}): {tie_block}\n")

rng = np.random.RandomState(0)
orders = {
    "impressions desc (declared)": df.sort_values(["baseline_score","impressions_90d"], ascending=[False,False]),
    "impressions asc":             df.sort_values(["baseline_score","impressions_90d"], ascending=[False,True]),
    "random":                      df.assign(r=rng.rand(len(df))).sort_values(["baseline_score","r"], ascending=[False,False]),
}
print(f"{'tie-break':<30}" + "".join(f"p@{k:<6}" for k in [10,20,50,100]))
for name, o in orders.items():
    print(f"{name:<30}" + "".join(f"{o.is_declining_label.head(k).mean():<8.3f}" for k in [10,20,50,100]))
print(f"\nbase rate {BASE_RATE:.3f} -- every tie-break clears it, so the SIGNAL is real;")
print("the exact precision@10 is not. Report the band, not the best number in it.")

pages tied at the maximum score (0.23): 1348

tie-break                     p@10    p@20    p@50    p@100   
impressions desc (declared)   0.800   0.800   0.860   0.880   
impressions asc               0.600   0.750   0.760   0.730   
random                        0.600   0.750   0.800   0.750   

base rate 0.542 -- every tie-break clears it, so the SIGNAL is real;
the exact precision@10 is not. Report the band, not the best number in it.


### Leakage check — no future-window or label-derived inputs

The label is computed from `impressions_last_30d` vs `impressions_prev_30d`. So the outcome window
and its own baseline window are both banned, along with the columns derived from them and their
click/session siblings. The rule's inputs are asserted against that list rather than eyeballed.

In [11]:
BANNED = ["trend_direction", "trend_pct",
          "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

leaked = sorted(set(RULE_INPUTS) & set(BANNED))
assert not leaked, f"LEAK: rule uses banned columns {leaked}"
print("rule inputs:", RULE_INPUTS)
print("banned:     ", BANNED)
print("PASS -- no future-window or label-derived column feeds the score.\n")

scored_cols = set(OUT_COLS) & set(BANNED)
assert not scored_cols, f"LEAK: exported queue carries banned columns {sorted(scored_cols)}"
print("PASS -- the exported queue carries none of them either.")

print("\nProduct flags (health_score / priority_score / action_type / refresh_tier) are not shipped")
print("in this dataset, so they cannot leak in:",
      sorted(c for c in ["health_score","priority_score","action_type","refresh_tier"] if c in df.columns) or "none present.")

print("\nDeclared and accepted, not hidden:")
print(" - label_eligible (impressions_prev_30d > 0) was used ONLY to diagnose Signal 1. It is not")
print("   a rule input, not an eligibility gate, and not in the exported queue. Every precision")
print("   number above is computed on all 30,000 rows.")
print(" - impressions_90d spans the label's own window. It is a LEVEL; the label is a DIRECTION,")
print("   and a level does not determine a direction. It enters the score only as a floor and a")
print("   tie-break -- and the rejected variant above shows what happens when it does more.")
print(" - tier_median_ctr is computed across the whole snapshot: a transparent global statistic,")
print("   no fitted weights, no label anywhere in it.")

rule inputs: ['avg_position', 'position_tier', 'ctr', 'impressions_90d']
banned:      ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
PASS -- no future-window or label-derived column feeds the score.

PASS -- the exported queue carries none of them either.

Product flags (health_score / priority_score / action_type / refresh_tier) are not shipped
in this dataset, so they cannot leak in: none present.

Declared and accepted, not hidden:
 - label_eligible (impressions_prev_30d > 0) was used ONLY to diagnose Signal 1. It is not
   a rule input, not an eligibility gate, and not in the exported queue. Every precision
   number above is computed on all 30,000 rows.
 - impressions_90d spans the label's own window. It is a LEVEL; the label is a DIRECTION,
   and a level does not determine a direction. It enters the score only as a floor and a
   tie-break -- and the rejected variant

### What this does to the lane

The queue looks like it indicts particular generation models. It doesn't — and the flag rates say
why.

In [12]:
by_model = (df.assign(is_flagged=flagged)
              .groupby(df.model_used.fillna("(no model recorded)"))
              .agg(n=("is_flagged","size"), flag_rate=("is_flagged","mean"))
              .round(3).sort_values("flag_rate", ascending=False))
print("share of each model's pages that the rule flags:")
print(by_model.to_string())

share of each model's pages that the rule flags:
                            n  flag_rate
model_used                              
(no model recorded)      5733      0.398
gemini-3-flash-preview  13271      0.248
gemini-2.5-flash         3665      0.207
gpt-5-mini               1598      0.162
gpt-4o-mini              4981      0.143
unknown                   752      0.137


The most-flagged group, by a wide margin, is the one where **no model was recorded at all**
(39.8%, n=5,733) — a metadata-availability artifact, not a model. Below it the recorded models
sit between 13.7% and 24.8%, and `content_type` is nearly collinear with `model_used` on this
slice, so even that ordering is not readable as a model effect.

Which is the same finding as ML-03 (adding `model_used` moved AUC by +0.0042) arriving through a
completely different instrument. A rule that never mentions `model_used` still produces a queue
whose model composition looks like an indictment. That's the confound writing the answer, and it's
the reason this baseline reports a reason code instead of a culprit.

## 5. Self-check

- [x] Two signals checked with visible bucket tables and n, both behind real FlyRank flags —
      staleness **FALSE**, CTR-below-tier-median **CONFIRMED**
- [x] The negative is explained, not buried: the staleness gradient is a labelability artifact,
      and killing it is what redirected the rule
- [x] One rule: score (`tier_median_ctr − ctr`), one reason code
      (`ctr_below_position_tier_median`), one action label (`review_title_and_meta`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`;
      metrics receipt committed to `work/outputs/baseline_action_score_metrics.json`
- [x] Twenty reviewed rows (card asks ten; the skeleton header and `building-baselines` both say
      twenty), each with the action, reason code, confidence note, why it's there, and what would
      make it wrong
- [x] Weak picks found: the zero-click block, the composition mistake that ranks below random,
      and a 1,348-row tie that decides precision@10 by tie-break
- [x] No future-window or label-derived inputs — asserted, not eyeballed
- [x] Runs top to bottom with no errors; fully deterministic (no seeds needed)
- [x] No client names, URLs, or private queries anywhere
- [x] Careful words throughout: observed, measured, directional, decision-support
- [x] Lane confirmed for the lock: Freestyle — "Confound or Cause?"

**Handing forward to Week 5.** The frozen number to beat: **precision@50 = 0.860** against a
**0.542** base rate, on all 30,000 rows. Because the top of the queue is one tie block, both
headline figures are reported as tie-break bands — **precision@10 = 0.600–0.800**, **precision@50
= 0.760–0.860** — and the model has to clear the top of the band, not the bottom. The model has to beat the top of that band on the same slice and
the same label, and it has to survive the same question this baseline just failed: is the queue
actionable, or is it ranking a measurement artifact?